This is the code that does all the preprocessing for the clavata expression file. Requires the clavata expression file, filtered reciprocal blast match file (clavata_4683_match), clavata annotation file and Orthogroup.tsv. It outputs two expression files, summed_clavata_expression and max_clavata_expression which are tsv, TPM normalized, and display orthogroups instead of genes.

In [ ]:
# Creates translation file between the GeneID used in the expression file,
# and the GeneID used in the orthogroups file.

import csv

# File paths
protein_match_file = '/content/drive/MyDrive/expression/clavata_4683_match.txt'
annotation_file = '/content/drive/MyDrive/expression/clavata_annotation.txt'
output_file = '/content/drive/MyDrive/expression/clavata_4683_translation.tsv'

query_to_gene = {}
with open(annotation_file, 'r') as anno_file:
    for line in anno_file:
        line = line.strip()
        if not line:
            continue
        columns = line.split('\t')
        if len(columns) < 2:
            continue
        geneID, queryID = columns[0], columns[1]
        query_to_gene[queryID] = geneID

# Process clavata_protein_match.txt and update queryID to geneID
updated_rows = []
with open(protein_match_file, 'r') as match_file:
    for line in match_file:
        line = line.strip()
        if not line:
            continue
        columns = line.split()
        if len(columns) < 2:
            continue
        queryID, targetID = columns[0], columns[1]
        if queryID in query_to_gene:
            geneID = query_to_gene[queryID]
            updated_rows.append([geneID, targetID])

# Write the updated rows to a new file
with open(output_file, 'w') as out_file:
    for row in updated_rows:
        out_file.write('\t'.join(row) + '\n')

In [ ]:
# Final translation step, translation file between the GeneID in expression
# and the corresponding orthogroup.

import csv

# Clavata gene ID to lascID
file_path_1 = '/content/drive/MyDrive/expression/clavata_4683_translation.tsv'

# List of orthogroups, with only clavata entries
file_path_2 = "/content/drive/MyDrive/expression/Orthogroups.tsv"

# Load the mapping from the first file (GeneID -> LascID)
format1_to_format2 = {}
with open(file_path_1, 'r') as file1:
    reader = csv.reader(file1, delimiter='\t')
    for row in reader:
        format1_to_format2[row[0]] = row[1]

# Load the mapping from the second file (LascID -> orthogroup)
format2_to_orthogroup = {}
with open(file_path_2, 'r') as file2:
    reader = csv.reader(file2, delimiter='\t')
    for row in reader:
        orthogroup = row[0]
        proteins_format2 = row[114].split(', ')
        for protein in proteins_format2:
            format2_to_orthogroup[protein] = orthogroup

# Create the output (geneID -> orthogroup)
output_rows = []
for protein_format1, protein_format2 in format1_to_format2.items():
    orthogroup = format2_to_orthogroup.get(protein_format2)  # None if not found
    if orthogroup:  # Skip if orthogroup is None
        output_rows.append([protein_format1, orthogroup])

# Write the output to a new TSV file
with open('/content/drive/MyDrive/expression/clavata_to_OG.tsv', 'w', newline='') as output_file:
    writer = csv.writer(output_file, delimiter='\t')
    writer.writerows(output_rows)


In [ ]:
# Renames all genes in expression.txt to the corresponding OG

import csv

# Load the mapping from the output TSV file (geneID -> orthogroup)
format1_to_orthogroup = {}
with open('/content/drive/MyDrive/expression/clavata_to_OG.tsv', 'r') as output_file:
    reader = csv.reader(output_file, delimiter='\t')
    next(reader)  # Skip the header
    for row in reader:
        format1_to_orthogroup[row[0]] = row[1]

# Using the clavata expression file - translates the first column ID
updated_rows = []
with open('/content/drive/MyDrive/expression/expression.txt', 'r') as input_file:
    reader = csv.reader(input_file, delimiter='\t')
    header = next(reader)  # Read and store the header row
    header[0] = "Orthogroup"  # Rename the first column
    updated_rows.append(header)  # Add the modified header row to the updated rows

    for row in reader:
        format1_name = row[0]
        orthogroup = format1_to_orthogroup.get(format1_name, "NA")
        updated_row = [orthogroup] + row[1:]
        if orthogroup != "NA":
            updated_rows.append(updated_row)

# Write the modified data to a new file
with open('/content/drive/MyDrive/expression/expression_clavata_OG.txt', 'w', newline='') as output_file:
    writer = csv.writer(output_file, delimiter='\t')
    writer.writerows(updated_rows)


In [ ]:
# Uses pandas to filter the clavata expression file containing orthogroups
# so that only one entry per orthogroup remains, filtering can be altered.
# This creates 2 filtered files with summed orthogroup genes & the max values

import pandas as pd

# Load the data into a Pandas DataFrame
df = pd.read_csv('/content/drive/MyDrive/expression/expression_clavata_OG.txt', sep='\t')

# Function to select how the composite orthogroup entry is to be constructed,
def select_sums(group):
    return group.sum(numeric_only=True)  # Sums

def select_max(group):
    return group.max(numeric_only=True)  # Max

# Apply grouping and sum
merged_df = df.groupby('Orthogroup', as_index=False).apply(select_sums)

# Write the merged data to a new file
merged_df.to_csv('/content/drive/MyDrive/expression/summed_clavata_expression.txt', sep='\t', index=False)

# Apply grouping and select maximum values
merged_df = df.groupby('Orthogroup', as_index=False).apply(select_max)

# Write the merged data to a new file
merged_df.to_csv('/content/drive/MyDrive/expression/max_clavata_expression.txt', sep='\t', index=False)



<ipython-input-62-06582eaf023c>:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  merged_df = df.groupby('Orthogroup', as_index=False).apply(select_sums)
<ipython-input-62-06582eaf023c>:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  merged_df = df.groupby('Orthogroup', as_index=False).apply(select_max)
